# Analyses GPU du rapport (TabPFN et TabM)

Calcule, dans l'ordre, les prédictions brutes lues par `analyses/tabm_tabpfn_gpu.py` et
`analyses/courbes_apprentissage.py` :
1. **TabPFN : nombre de passages et cible** sur le pli 1 (fichiers `B_*`) ;
2. **courbes d'apprentissage** de TabPFN et TabM avec 1 000 à 30 000 lignes (fichiers `C_*`) ;
3. **recherche de 16 configurations de TabM** + configuration de départ, 63 000 / 21 000 lignes (fichiers `R_*`) ;
4. **meilleure configuration TabM** en validation croisée, test interne et test (fichiers `T_*`).

Les sorties fournies sont déjà dans `resultats/gpu/` ; ce notebook sert à les recalculer.
**Avant de lancer :** GPU et secret `TABPFN_TOKEN` (comme `2_modeles_gpu.ipynb`). **Ensuite :** *Tout exécuter* ;
`analyses_gpu.zip` est téléchargé à la fin : en extraire le contenu dans `resultats/gpu/`.

In [ ]:
import os, sys, json, time, subprocess, importlib.metadata
from pathlib import Path
from google.colab import userdata, drive, files

# Clé d'API Prior Labs (licence TabPFN acceptée sur ux.priorlabs.ai), rangée dans les secrets Colab.
jeton = userdata.get('TABPFN_TOKEN')
assert isinstance(jeton, str) and jeton.strip(), 'Le secret TABPFN_TOKEN est absent ou vide.'
os.environ['TABPFN_TOKEN'] = jeton.strip(); os.environ['TABPFN_NO_BROWSER'] = '1'; del jeton

# Versions de TabPFN et TabM du projet ; on garde torch, numpy, pandas, scipy et scikit-learn de Colab.
BASE = ['torch', 'numpy', 'pandas', 'scipy', 'scikit-learn']
Path('/content/contraintes.txt').write_text('\n'.join(f'{n}=={importlib.metadata.version(n)}' for n in BASE) + '\n')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'tabpfn==9.1.0', 'tabm==0.0.3', 'rtdl_num_embeddings==0.0.12',
                'lightgbm', 'catboost', '--constraint', '/content/contraintes.txt'], check=True)
import torch
assert torch.cuda.is_available(), 'Pas de GPU : Exécution → Modifier le type d’exécution → GPU.'
print('GPU :', torch.cuda.get_device_name(0))

# Le dépôt contient le code et les données.
if not Path('/content/depot').is_dir():
    subprocess.run(['git', 'clone', '-q', 'https://github.com/Hamdane11/performances-powerlifting', '/content/depot'], check=True)
sys.path.insert(0, '/content/depot/src')
drive.mount('/content/drive')

In [ ]:
import numpy as np
import pandas as pd
from powerlifting.data import charger, configuration, decoupage
from powerlifting.models import predict_model

CFG = configuration()
train, test = charger()
d = decoupage(len(train))
X_all, y_all = train.drop(columns=['TotalKg']), train['TotalKg'].to_numpy(float)
print(len(d.dev), 'lignes de développement,', len(d.test_interne), 'de test interne,', len(test), 'de test')

COLONNES = list(CFG['variables'])

def preparer(frame):
    x = frame.loc[:, COLONNES].copy().reset_index(drop=True)
    x.loc[x['Age'] < 5, 'Age'] = np.nan
    for c in COLONNES[3:]:
        x[c] = x[c].fillna('Unknown').astype(str)
    return x

def tabpfn_8_passages(contexte, y_contexte, requetes, passages=8, exposant=2/3, bloc=15000):
    """TabPFN 3.5 : 8 passages, cible divisée par (poids/80)^(2/3), espérance de la loi prédite."""
    from tabpfn import TabPFNRegressor
    from tabpfn.constants import ModelVersion
    xc, xq = preparer(contexte), preparer(requetes)
    echelle = lambda x: (x['BodyweightKg'].to_numpy(float) / 80.0) ** exposant
    m = TabPFNRegressor.create_default_for_version(
        ModelVersion.V3_5, n_estimators=passages, device='cuda', random_state=2026,
        fit_mode='fit_preprocessors', memory_saving_mode=True, n_preprocessing_jobs=1,
        categorical_features_indices=list(range(3, 16)), inference_config={'SUBSAMPLE_SAMPLES': None},
        tuning_config=None, average_before_softmax=False, ignore_pretraining_limits=False)
    m.fit(xc, np.asarray(y_contexte, float) / echelle(xc))
    sq, p = echelle(xq), np.empty(len(xq))
    for a in range(0, len(xq), bloc):
        p[a:a + bloc] = np.asarray(m.predict(xq.iloc[a:a + bloc], output_type='mean'), float) * sq[a:a + bloc]
    del m; torch.cuda.empty_cache()
    return np.maximum(p, 0.0)

from sklearn.model_selection import train_test_split
from powerlifting.models import tabm as tabm_mod

SORTIE = Path('/content/drive/MyDrive/powerlifting/analyses_gpu')
SORTIE.mkdir(parents=True, exist_ok=True)
dev, folds, hold = d.dev, d.plis, d.test_interne
fit_idx, select_idx = train_test_split(dev, test_size=0.25, random_state=43)
contexte_pli1, requetes_pli1 = dev[folds != 1], dev[folds == 1]
np.savez_compressed(SORTIE / 'indices.npz', dev=dev, hold=hold, folds=folds, fit_idx=fit_idx, select_idx=select_idx)
TABM_GRAND, TABM_PETIT = CFG['modeles']['tabm'], CFG['modeles']['tabm_allom']

def tabpfn(nom, ctx_frame, ctx_y, qry_frame, cible='allom', passages=8):
    fichier = SORTIE / f'{nom}.npz'
    if fichier.is_file():
        print('déjà fait :', nom); return
    t = time.time()
    p = tabpfn_8_passages(ctx_frame, ctx_y, qry_frame, passages=passages, exposant=0.0 if cible == 'kg' else 2 / 3)
    np.savez_compressed(fichier, prediction=p, secondes=time.time() - t)
    print(f'{nom} : {(time.time() - t) / 60:.1f} min')

def tabm_fixe(nom, config, ctx_frame, ctx_y, qry_frame):
    fichier = SORTIE / f'{nom}.npz'
    if fichier.is_file():
        print('déjà fait :', nom); return
    t = time.time()
    f = tabm_mod.fit_candidate(ctx_frame.reset_index(drop=True), np.asarray(ctx_y, float), dict(config))
    np.savez_compressed(fichier, prediction=tabm_mod.predict_candidate(f, qry_frame.reset_index(drop=True)),
                        secondes=time.time() - t)
    torch.cuda.empty_cache(); print(f'{nom} : {(time.time() - t) / 60:.1f} min')

def tabm_selection(nom, config, a_frame, a_y, v_frame, v_y, qry_frame=None):
    fichier = SORTIE / f'{nom}.npz'
    if fichier.is_file():
        print('déjà fait :', nom); return dict(np.load(fichier, allow_pickle=False))
    t = time.time()
    f = tabm_mod.fit_selection(a_frame.reset_index(drop=True), np.asarray(a_y, float),
                               v_frame.reset_index(drop=True), np.asarray(v_y, float), dict(config))
    sortie = dict(best_epoch=f['best_epoch'], best_score=f['best_score'], secondes=time.time() - t,
                  prediction_validation=tabm_mod.predict_candidate(f, v_frame.reset_index(drop=True)))
    if qry_frame is not None:
        sortie['prediction'] = tabm_mod.predict_candidate(f, qry_frame.reset_index(drop=True))
    np.savez_compressed(fichier, **sortie)
    torch.cuda.empty_cache(); print(f"{nom} : {f['best_score']:.3f} kg, {f['best_epoch']} époques")
    return sortie

## 1. TabPFN : nombre de passages et cible (pli 1)

In [ ]:
Xc, yc, Xq = X_all.iloc[contexte_pli1], y_all[contexte_pli1], X_all.iloc[requetes_pli1]
for n in (1, 2, 4, 8, 16):
    tabpfn(f'B_allom_{n}', Xc, yc, Xq, 'allom', n)
for n in (4, 8):
    tabpfn(f'B_kg_{n}', Xc, yc, Xq, 'kg', n)

## 2. Courbes d'apprentissage (pli 1 prédit, sous-échantillons emboîtés du contexte)

In [ ]:
ordre = np.random.default_rng(2026).permutation(len(contexte_pli1))
for taille in (1000, 3000, 10000, 30000):
    lignes = contexte_pli1[np.sort(ordre[:taille])]
    tabpfn(f'C_tabpfn_{taille}', X_all.iloc[lignes], y_all[lignes], Xq, 'allom', 8)
    coupe = int(0.9 * taille)          # TabM : 90 % pour apprendre, 10 % pour l'arrêt anticipé
    a, v = lignes[:coupe], lignes[coupe:]
    tabm_selection(f'C_tabm_{taille}', TABM_GRAND, X_all.iloc[a], y_all[a], X_all.iloc[v], y_all[v], Xq)

## 3. Recherche d'hyperparamètres de TabM (63 000 / 21 000 lignes, arrêt anticipé)

In [ ]:
rng = np.random.default_rng(7)
ESPACE = dict(n_blocks=[2, 3, 4], d_block=[256, 384, 512], k=[16, 32], lr=[5e-4, 1e-3, 2e-3],
              dropout=[0.0, 0.1, 0.2], n_bins=[32, 64, 96], d_embedding=[8, 16, 24],
              weight_decay=[1e-4, 3e-4, 1e-3], exponent=[0.0, 2.0 / 3.0])
configs = [dict(TABM_GRAND, name='tabm_ref')]
for i in range(16):
    c = dict(TABM_GRAND, name=f'tabm_r{i:02d}')
    for cle, valeurs in ESPACE.items():
        c[cle] = valeurs[int(rng.integers(len(valeurs)))]
    configs.append(c)
for c in configs:
    c = {k: v for k, v in c.items() if k != 'epochs'}
    c['max_epochs'], c['patience'] = 160, 20
    tabm_selection(f"R_{c['name']}", c, X_all.iloc[fit_idx], y_all[fit_idx], X_all.iloc[select_idx], y_all[select_idx])
resume = []
for c in configs:
    r = np.load(SORTIE / f"R_{c['name']}.npz")
    resume.append({**{k: c[k] for k in ['name'] + list(ESPACE)}, 'rmse_validation': float(r['best_score']),
                   'epoques': int(r['best_epoch'])})
resume = pd.DataFrame(resume).sort_values('rmse_validation')
resume.to_csv(SORTIE / 'R_resume.csv', index=False)
print(resume.head(8).to_string(index=False))

## 4. Meilleure configuration TabM : validation croisée, test interne et test

In [ ]:
meilleure = resume.iloc[0]
cfg_best = next(dict(c) for c in configs if c['name'] == meilleure['name'])
cfg_best['epochs'] = int(meilleure['epoques'])
for k in range(1, 6):
    a, q = dev[folds != k], dev[folds == k]
    tabm_fixe(f'T_oof_pli{k}', cfg_best, X_all.iloc[a], y_all[a], X_all.iloc[q])
tabm_fixe('T_test_interne', cfg_best, X_all.iloc[dev], y_all[dev], X_all.iloc[hold])
tabm_fixe('T_test_kaggle', cfg_best, X_all, y_all, test)
(SORTIE / 'T_config.json').write_text(json.dumps(cfg_best, default=str, indent=1))

In [ ]:
import shutil
files.download(shutil.make_archive('/content/analyses_gpu', 'zip', SORTIE))